# 📈 20일선과 60일선으로 신호 검증하기 — Student

삼성전자 가격 데이터에서 이동평균, 교차 신호, 다음 날 포지션을 한 흐름으로 확인합니다.

> **🖥️ 환경 설정**
>
> - **로컬**: 교수자가 제공한 pyproject.toml + uv.lock을 폴더에 두고 터미널에서 uv sync 실행 → uv run jupyter lab

> **⌨️ 단축키 안내**
>
> | 단축키 | 동작 |
> |--------|------|
> | Shift + Enter | 셀 실행 후 다음 셀 이동 |
> | Esc → A | 위에 셀 삽입 |
> | Esc → B | 아래에 셀 삽입 |
> | Esc → DD | 셀 삭제 |

In [ ]:
# 🖥️ 환경 설정
# - 로컬: 터미널에서 uv sync 실행 (pyproject.toml + uv.lock 동봉)
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

# 🎛️ 파라미터
code = "005930"
start_date = "2024-06-25"
end_date = "2026-06-24"
db_path = Path("data/market_data.db")

## 🎯 데이터 불러오기 — SQL은 CSV를 대신할 뿐

[→ §100 강의노트](../notes/FD-05-100__second-window.md)

`data/market_data.db`는 `extend_market_data.py`가 미리 만들어 둔 삼성전자 2년치(2024-06-25~2026-06-24, 485행) 가격 테이블입니다. `pd.read_csv` 대신 `pd.read_sql_query`로 필요한 열만 꺼내는 것 외에는 다르지 않습니다.

In [ ]:
# 🛠️ §500-1 SQL로 불러오기
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(
    "SELECT date, close, daily_return FROM returns WHERE code=? AND date BETWEEN ? AND ?",
    conn, params=(code, start_date, end_date), parse_dates=["date"], index_col="date",
)
print(df.head(3))
# 예상 출력: 2024-06-25부터 시작하는 삼성전자 3행 (date가 인덱스)

## 📏 이동평균 — NaN은 모자란 창의 표시

[→ §100 강의노트](../notes/FD-05-100__second-window.md)

20일선·60일선을 만들기 전에 작은 수열로 워밍업 구간을 먼저 예측합니다. 앞 59행의 NaN은 오류가 아니라 60개의 가격이 아직 모이지 않았다는 뜻입니다.

In [ ]:
# 📌 §100-1 평균은 늦게 도착한다
# 코드를 작성해주세요 👇
# 예상 출력: 3번째 값부터 평균이 생김

In [ ]:
# 📌 §100-2 평균은 늦게 도착한다
# 코드를 작성해주세요 👇
# 예상 출력: 59

### 📊 차트 1 — 종가와 두 이동평균선

앞의 §100-2로 만든 `ma20`·`ma60`을 실제 2년치 삼성전자 종가 위에 겹쳐 그립니다.

In [ ]:
# 🛠️ §500-2 차트 — 종가 + ma20 + ma60
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df.index, df["close"], color="lightgray", label="close")
ax.plot(df.index, df["ma20"], color="green", label="ma20")
ax.plot(df.index, df["ma60"], color="orange", label="ma60")
ax.legend()
ax.set_title("Samsung Electronics: Close vs MA20/MA60")
plt.show()
# 예상 출력: close·ma20·ma60 3개 선 그래프

> 💬 왼쪽 끝 60일 구간에는 왜 주황색 ma60 선이 안 보일까요? 가격이 급하게 꺾이는 지점에서 두 이동평균선은 언제 방향을 바꾸나요 — 가격보다 먼저인가요, 나중인가요?

## 🔁 signal·cross — 상태와 전환을 분리하기

[→ §200 강의노트](../notes/FD-05-200__cross-as-hypothesis.md)

signal은 매일의 상태이고 cross는 상태가 바뀐 날입니다. 노트 §200 코드를 그대로 재현합니다.

In [ ]:
# 📌 §200-1 signal은 상태, cross는 날
# 코드를 작성해주세요 👇
# 예상 출력: signal·cross·dead_cross 열 생성

In [ ]:
# 📌 §200-2 과거에 몇 번 맞았나 — whipsaw 세기
# 코드를 작성해주세요 👇
# 예상 출력: 2 1 2 0

### 📊 차트 2 — 골든크로스 / 데드크로스 표시

같은 차트 위에 `cross`(▲ 골든크로스)와 `dead_cross`(▼ 데드크로스)가 난 날을 표시합니다.

In [ ]:
# 🛠️ §500-3 차트 — 골든/데드크로스 마커
golden = df.index[df["cross"]]
dead = df.index[df["dead_cross"]]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df.index, df["close"], color="lightgray", label="close")
ax.plot(df.index, df["ma20"], color="green", label="ma20")
ax.plot(df.index, df["ma60"], color="orange", label="ma60")
ax.scatter(golden, df.loc[golden, "close"], marker="^", color="green", s=120, zorder=5, label="golden cross")
ax.scatter(dead, df.loc[dead, "close"], marker="v", color="red", s=120, zorder=5, label="dead cross")
ax.legend()
ax.set_title("Golden Cross vs Dead Cross")
plt.show()
# 예상 출력: 초록 삼각형(golden cross) 2개, 빨강 역삼각형(dead cross) 1개

> 💬 골든크로스는 2번, 데드크로스는 1번 표시됩니다. 두 번의 골든크로스 뒤 20거래일 동안 가격은 실제로 올랐나요? 데드크로스 뒤에는 어땠나요?

## ⏱️ position·전략수익률 — 내일 보유한다

[→ §300 강의노트](../notes/FD-05-300__signal-tomorrow.md)

오늘 장 마감 뒤에 신호를 확인했다면 실제 보유는 다음 거래일부터입니다. `shift(1)`을 넣은 결과와 뺀 결과를 나란히 계산해 시간 누수를 눈으로 비교합니다.

In [ ]:
# 📌 §300-1 position은 하루 늦게 켜진다
# 코드를 작성해주세요 👇
# 예상 출력: position은 다음 거래일부터 1.0

In [ ]:
# 📌 §300-2 한 줄을 빼면 과거가 가짜 대박이 된다
# 코드를 작성해주세요 👇
# 예상 출력: 442.17% 445.77%
# 부호는 크로스 당일 등락에 따라 달라짐

### 📊 차트 3 — shift를 빼면 어느 날의 수익이 섞이나

두 전략수익률의 차이(누락 − 정답)를 날짜별 막대로 그립니다. 0이 아닌 날만이 미래 정보가 새어 든 날입니다.

In [ ]:
# 🛠️ §500-4 차트 — 새어 든 수익의 날짜와 부호
leak = df["strategy_return_wrong"] - df["strategy_return_correct"]   # 누락 버전이 더 가져간 수익
leak_days = leak[leak.fillna(0) != 0]
print(leak_days.round(4))

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(leak_days.index, leak_days, width=6, color=["#27AE60" if v > 0 else "#C0392B" for v in leak_days])
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("Leaked return on cross days (missing shift - correct)")
plt.show()
# 예상 출력: 크로스가 난 날들만 막대가 생기고, 부호가 날마다 다름

> 💬 막대가 생긴 날짜를 §200-1의 크로스 날짜와 비교해 보세요. 막대가 모두 위를 향하나요? 누락 버전이 이번에 더 크게 나온 것이 규칙인지, 이 종목·이 기간의 우연인지 한 문장으로 적어 보세요.

## 📝 가설 한 줄 — 신호를 약속이 아니라 검증 대상으로

[→ §200 강의노트](../notes/FD-05-200__cross-as-hypothesis.md)

마지막으로 자신의 규칙을 한 문장으로 적습니다. 신호, 행동 시점, 확인할 결과를 함께 적어야 나중에 반증 가능한 가설이 됩니다.

In [ ]:
# 🛠️ §500-5 가설 한 줄
# 코드를 작성해주세요 👇
# 예상 출력: 내가 정한 문장이 그대로 출력됨